<a href="https://colab.research.google.com/github/hessenali383/cat_dog_classifier_DEBI/blob/main/Copy_of_notebook57cff8cfcd_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cat vs Dog Image Classification

Comparing three models on the [Cats and Dogs dataset](https://www.kaggle.com/datasets/tongpython/cat-and-dog):

- **ANN** — simple fully connected network, baseline model
- **CNN (Fine-tuned)** — 4 conv blocks + dropout + GlobalAveragePooling
- **ResNet50 (Transfer Learning)** — pretrained on ImageNet, best performance

**Result:** ANN < Fine-tuned CNN < ResNet50

In [ ]:
import kagglehub
path = kagglehub.dataset_download('tongpython/cat-and-dog')

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cat-and-dog' dataset.
Path to dataset files: /kaggle/input/cat-and-dog


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.resnet50 import preprocess_input



In [ ]:
path = "/kaggle/input/cat-and-dog"

# Define paths to training and validation directories


In [ ]:
train_dir = os.path.join(path, "training_set/training_set")
validation_dir = os.path.join(path, "test_set/test_set")

# show the number of images in each class
num_cats_train = len(os.listdir(os.path.join(train_dir, "cats")))
num_dogs_train = len(os.listdir(os.path.join(train_dir, "dogs")))
num_cats_validation = len(os.listdir(os.path.join(validation_dir, "cats")))
num_dogs_validation = len(os.listdir(os.path.join(validation_dir, "dogs")))

print(f"Number of training cat images: {num_cats_train}")
print(f"Number of training dog images: {num_dogs_train}")
print(f"Number of validation cat images: {num_cats_validation}")
print(f"Number of validation dog images: {num_dogs_validation}")

# Show size of a sample image
sample_image_path = os.path.join(train_dir, "cats", os.listdir(os.path.join(train_dir, "cats"))[0])
sample_image = tf.keras.preprocessing.image.load_img(sample_image_path)
print(f"Sample image size: {sample_image.size}")

Number of training cat images: 4001
Number of training dog images: 4006
Number of validation cat images: 1012
Number of validation dog images: 1013
Sample image size: (500, 332)


# define function to process the dataset


In [ ]:
def process_dataset(train_dir, validation_dir, img_size=(224,224), batch_size=64):

    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
    )

    validation_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input
    )

    train_generator = train_datagen.flow_from_directory(
        train_dir, target_size=img_size,
        batch_size=batch_size, class_mode="binary"
    )
    validation_generator = validation_datagen.flow_from_directory(
        validation_dir, target_size=img_size,
        batch_size=batch_size, class_mode="binary"
    )
    return train_generator, validation_generator

# build the ANN model

In [ ]:
def build_model(input_shape=(224, 224, 3)):
    model = models.Sequential()

    model.add(layers.Flatten(input_shape = input_shape))

    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model

# train the model


In [ ]:
def train_model(model, train_generator, validation_generator, epochs=20):
    history = model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // train_generator.batch_size,
        validation_data=validation_generator,
        validation_steps=validation_generator.samples // validation_generator.batch_size,
        epochs=epochs,
    )
    return history

In [ ]:
# move model to GPU if available
device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"

In [ ]:
print(device)

GPU


In [ ]:
# start the training process
train_generator, validation_generator = process_dataset(train_dir,validation_dir)
model_ANN = build_model()
history = train_model(model_ANN, train_generator, validation_generator)

Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 85s 654ms/step - accuracy: 0.5317 - loss: 506.1620 - val_accuracy: 0.5459 - val_loss: 245.8838
Epoch 2/20
  1/125 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.5312 - loss: 217.0219

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - accuracy: 0.5312 - loss: 217.0219 - val_accuracy: 0.5514 - val_loss: 227.0450
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 36s 286ms/step - accuracy: 0.5713 - loss: 155.3862 - val_accuracy: 0.5388 - val_loss: 118.0818
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.4688 - loss: 118.6622 - val_accuracy: 0.5554 - val_loss: 107.2784
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 35s 282ms/step - accuracy: 0.5906 - loss: 86.7273 - val_accuracy: 0.5570 - val_loss: 75.3594
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.6250 - loss: 65.7567 - val_accuracy: 0.5509 - val_loss: 77.4548
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 35s 276ms/step - accuracy: 0.5967 - loss: 66.8067 - val_accuracy: 0.5665 - val_loss: 61.8330
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.5312 - loss: 67.5294 - val_accuracy: 0.5680 - val_loss: 62.2781
Epoch 9/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 75s 277ms/step - accuracy: 0.5909 - loss: 

# fine tuning the hyperparameters

## This Code of fine tuning was made by Claude Ai

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PATCH CELL — شغّل هذا بعد الكود الأصلي مباشرةً
#  الإصلاح: drop_remainder=True عبر tf.data لتجنب batch mismatch
#  مع MirroredStrategy (T4 x2)
# ══════════════════════════════════════════════════════════════════

def make_tf_dataset(flow, samples=None, batch_size=BATCH_SIZE,
                    img_size=IMG_SIZE, repeat=False):
    """
    يحوّل ImageDataGenerator flow إلى tf.data.Dataset
    مع drop_remainder=True لإصلاح مشكلة MirroredStrategy.
    """
    ds = tf.data.Dataset.from_generator(
        lambda: flow,
        output_signature=(
            tf.TensorSpec(shape=(None, *img_size, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(None,),              dtype=tf.float32),
        )
    ).unbatch()                                   # فك كل batch من الـ generator

    if samples:
        ds = ds.take(samples)                     # حدّ عدد الصور للبحث السريع

    ds = ds.batch(batch_size, drop_remainder=True) # ← الإصلاح الأساسي

    if repeat:
        ds = ds.repeat()

    return ds.prefetch(tf.data.AUTOTUNE)


# ── أعِد بناء الـ generators الأصلية (بدون تغيير) ────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator as IDG

_train_idg = IDG(rescale=1./255, horizontal_flip=True, zoom_range=0.1)
_val_idg   = IDG(rescale=1./255)

_train_flow = _train_idg.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", shuffle=True)

_val_flow = _val_idg.flow_from_directory(
    VALIDATION_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)

# ── حوّلهم لـ tf.data مع drop_remainder ──────────────────────────
train_ds = make_tf_dataset(_train_flow, samples=SAMPLES,
                           batch_size=BATCH_SIZE, repeat=True)
val_ds   = make_tf_dataset(_val_flow,   samples=None,
                           batch_size=BATCH_SIZE, repeat=False)

steps_per_epoch = SAMPLES    // BATCH_SIZE
val_steps       = _val_flow.samples // BATCH_SIZE

print(f"✅ Datasets جاهزة | steps/epoch={steps_per_epoch} | val_steps={val_steps}")

# ══════════════════════════════════════════════════════════════════
#  حلقة البحث — نفس الكود الأصلي بالضبط لكن يمرّر train_ds/val_ds
# ══════════════════════════════════════════════════════════════════
results = []

for idx, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    label  = (f"LR={params['learning_rate']:.0e} | "
              f"blocks={params['conv_blocks']} | "
              f"filters={params['base_filters']} | "
              f"dense={params['dense_units']}")

    print(f"[{idx+1}/{len(combos)}]  {label}")

    tf.keras.backend.clear_session()

    model = build_model(
        conv_blocks   = params["conv_blocks"],
        base_filters  = params["base_filters"],
        dense_units   = params["dense_units"],
        learning_rate = params["learning_rate"],
    )

    history = model.fit(
        train_ds,
        steps_per_epoch  = steps_per_epoch,
        validation_data  = val_ds,
        validation_steps = val_steps,
        epochs           = EPOCHS,
        verbose          = 0,
    )

    final_val_loss = history.history["val_loss"][-1]
    final_val_acc  = history.history["val_accuracy"][-1]
    print(f"   → val_loss={final_val_loss:.4f}  val_acc={final_val_acc:.4f}\n")

    results.append({
        "params"   : params,
        "label"    : label,
        "history"  : history.history,
        "val_loss" : final_val_loss,
        "val_acc"  : final_val_acc,
    })

results.sort(key=lambda r: r["val_loss"])

# ══════════════════════════════════════════════════════════════════
#  الجرافات — نفس الكود الأصلي تماماً
# ══════════════════════════════════════════════════════════════════
LRs          = SEARCH_SPACE["learning_rate"]
epochs_range = range(1, EPOCHS + 1)

# ① Grid: Loss Curves by LR
fig, axes = plt.subplots(len(LRs), 2, figsize=(18, 5 * len(LRs)))
fig.suptitle("Training & Validation Loss — by Learning Rate",
             fontsize=16, fontweight="bold", y=1.01)

for row, lr in enumerate(LRs):
    ax_t = axes[row][0]; ax_v = axes[row][1]
    subset = [r for r in results if r["params"]["learning_rate"] == lr]

    ax_t.set_title(f"LR = {lr:.0e}  |  Train Loss",  fontsize=12)
    ax_v.set_title(f"LR = {lr:.0e}  |  Val Loss",    fontsize=12)

    for r in subset:
        lbl = (f"blk={r['params']['conv_blocks']} "
               f"flt={r['params']['base_filters']} "
               f"dns={r['params']['dense_units']}")
        ax_t.plot(epochs_range, r["history"]["loss"],     label=lbl, linewidth=1.8)
        ax_v.plot(epochs_range, r["history"]["val_loss"], label=lbl, linewidth=1.8, linestyle="--")

    for ax in [ax_t, ax_v]:
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(fontsize=7, loc="upper right"); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/loss_curves_by_lr.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ loss_curves_by_lr.png")

# ② Conv Blocks effect at best LR
best_lr    = results[0]["params"]["learning_rate"]
fig2, ax2s = plt.subplots(1, 2, figsize=(16, 5))
fig2.suptitle(f"Effect of Conv Blocks & Dense Units  (LR={best_lr:.0e})",
              fontsize=14, fontweight="bold")

for r in [r for r in results if r["params"]["learning_rate"] == best_lr]:
    lbl = (f"blk={r['params']['conv_blocks']} "
           f"dns={r['params']['dense_units']} "
           f"flt={r['params']['base_filters']}")
    ax2s[0].plot(epochs_range, r["history"]["loss"],     label=lbl, linewidth=1.8)
    ax2s[1].plot(epochs_range, r["history"]["val_loss"], label=lbl, linewidth=1.8, linestyle="--")

ax2s[0].set_title("Train Loss"); ax2s[1].set_title("Val Loss")
for ax in ax2s:
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/loss_curves_blocks.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ loss_curves_blocks.png")

# ③ Top-10 bar chart
top_n      = min(10, len(results))
top        = results[:top_n]
labels     = [f"LR={r['params']['learning_rate']:.0e}\nblk={r['params']['conv_blocks']} dns={r['params']['dense_units']}" for r in top]
val_losses = [r["val_loss"] for r in top]
val_accs   = [r["val_acc"]  for r in top]
x          = np.arange(top_n)

fig3, ax3 = plt.subplots(figsize=(16, 6))
bars = ax3.bar(x, val_losses,
               color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, top_n)),
               edgecolor="black", linewidth=0.7)

for bar, acc in zip(bars, val_accs):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
             f"acc={acc:.2%}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax3.set_xticks(x); ax3.set_xticklabels(labels, fontsize=8)
ax3.set_ylabel("Final Val Loss (↓ better)")
ax3.set_title("Top Hyperparameter Combos — Final Validation Loss",
              fontsize=14, fontweight="bold")
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/top_combos_bar.png", dpi=150, bbox_inches="tight")
plt.show(); print("✅ top_combos_bar.png")

# ④ Leaderboard
print("\n" + "═"*65)
print("  🏆  HYPERPARAMETER LEADERBOARD  (sorted by val_loss)")
print("═"*65)
for rank, r in enumerate(results, 1):
    p = r["params"]
    print(f"  #{rank:>2}  LR={p['learning_rate']:.0e} | blocks={p['conv_blocks']} | "
          f"filters={p['base_filters']} | dense={p['dense_units']}  "
          f"→  val_loss={r['val_loss']:.4f}  val_acc={r['val_acc']:.2%}")
print("═"*65)
print(f"\n🥇 Best: {results[0]['label']}")
print(f"   val_loss={results[0]['val_loss']:.4f}  |  val_acc={results[0]['val_acc']:.2%}")

# the resulit of fine tuning THE CNN will be:

* **add dropout layers = 0.25**
* **add 4th conv2d block**
* **change flatten layer to GlobalAveragePooling2D**
* **sustane the learning rate = 0.001 like the last learning rate**

In [ ]:
def build_model_2(input_shape=(224, 224, 3)):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(32, (3,3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 2
    model.add(layers.Conv2D(64, (3,3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 3
    model.add(layers.Conv2D(128, (3,3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 4
    model.add(layers.Conv2D(256, (3,3), activation="relu", padding="same"))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
train_generator, validation_generator = process_dataset(train_dir, validation_dir)

model = build_model_2()

history = model.fit(
    train_generator,
    steps_per_epoch  = len(train_generator),
    validation_data  = validation_generator,
    validation_steps = len(validation_generator),
    epochs = 20,
)

Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.
Epoch 1/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 65s 390ms/step - accuracy: 0.5239 - loss: 1.3730 - val_accuracy: 0.5467 - val_loss: 0.6716
Epoch 2/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 37s 291ms/step - accuracy: 0.5820 - loss: 0.6613 - val_accuracy: 0.6026 - val_loss: 0.6522
Epoch 3/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 38s 304ms/step - accuracy: 0.5978 - loss: 0.6520 - val_accuracy: 0.6036 - val_loss: 0.6582
Epoch 4/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 358ms/step - accuracy: 0.6289 - loss: 0.6335 - val_accuracy: 0.6327 - val_loss: 0.6455
Epoch 5/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 48s 380ms/step - accuracy: 0.6632 - loss: 0.6147 - val_accuracy: 0.5971 - val_loss: 0.6466
Epoch 6/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 48s 382ms/step - accuracy: 0.6821 - loss: 0.5899 - val_accuracy: 0.6891 - val_loss: 0.5897
Epoch 7/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 40s 319ms/step - accuracy: 0.7004 - loss: 0.5684 - val_accuracy: 0.7074 - val_loss: 0.5706

# Now build the model using transfer learning with Resnet50


In [ ]:
def build_transfer_learning_model(input_shape=(224, 224, 3)):
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_shape=input_shape
    )
    base_model.trainable = False  # Freeze the base model

    model = models.Sequential()
    model.add(base_model)

    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model

In [ ]:
train_generator, validation_generator = process_dataset(train_dir, validation_dir)

model2 = build_transfer_learning_model()

history = model2.fit(
    train_generator,
    steps_per_epoch  = len(train_generator),
    validation_data  = validation_generator,
    validation_steps = len(validation_generator),
    epochs = 20,
)

Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step
Epoch 1/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 85s 549ms/step - accuracy: 0.9651 - loss: 0.0895 - val_accuracy: 0.9886 - val_loss: 0.0361
Epoch 2/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 52s 413ms/step - accuracy: 0.9899 - loss: 0.0297 - val_accuracy: 0.9837 - val_loss: 0.0475
Epoch 3/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 46s 365ms/step - accuracy: 0.9930 - loss: 0.0191 - val_accuracy: 0.9891 - val_loss: 0.0368
Epoch 4/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 48s 379ms/step - accuracy: 0.9969 - loss: 0.0096 - val_accuracy: 0.9867 - val_loss: 0.0439
Epoch 5/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 52s 409ms/step - accuracy: 0.9980 - loss: 0.0066 - val_accuracy: 0.9901 - val_loss: 0.0431
Epoch 6/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 50s 395ms/step - accuracy: 0.9993 - loss: 0.0036 - val_accuracy: 0.9891 - val_loss: 0.0417
Epoch 7/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 356ms/step - accuracy: 0.9999 - los

In [ ]:
# now we can evaluate the model on the validation set
loss, accuracy = model2.evaluate(validation_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

32/32 ━━━━━━━━━━━━━━━━━━━━ 9s 266ms/step - accuracy: 0.9891 - loss: 0.0526
Validation Loss: 0.0526
Validation Accuracy: 0.9891


### PLOT THE Confusion matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


validation_datagen_final = ImageDataGenerator(preprocessing_function=preprocess_input)
# we need validation_datagen without shuffle to test
val_gen_no_shuffle = validation_datagen_final.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    batch_size=64,
    class_mode="binary",
    shuffle=False
)

predictions = model2.predict(val_gen_no_shuffle)
y_pred = (predictions > 0.5).astype(int).flatten()

y_true = val_gen_no_shuffle.classes

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Cat', 'Dog'])

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues)
plt.title("Confusion Matrix - ResNet50")
plt.show()

In [ ]:
# save the model
model2.save("cat_dog_model.h5")